# Import thư viện và Đọc dữ liệu gốc

In [ ]:
#Khai báo thư viện
import pandas as pd
import numpy as np

In [ ]:
#Đọc file và và chuyển dạng chuỗi (string) trước để tự kiểm soát việc ép kiểu
customer='/content/customers.csv'
df_raw = pd.read_csv(customer)
# , dtype=str, keep_default_na=True
df_raw.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


# Xem tổng quan

In [ ]:
#Đọc số dòng cột của file và xem kiểu dữ liệu của từng thuộc tính
df=df_raw.copy()
print("Số dòng và số cột",df.shape)
print("\nTên cột & dtype hiện tại (đọc dạng string để kiểm soát):")
print(df.dtypes)

Số dòng và số cột (121930, 7)

Tên cột & dtype hiện tại (đọc dạng string để kiểm soát):
customer_id             int64
zip                     int64
city                   object
signup_date            object
gender                 object
age_group              object
acquisition_channel    object
dtype: object


In [ ]:
pd.isnull(df).sum()

,0
customer_id,0
zip,0
city,0
signup_date,0
gender,0
age_group,0
acquisition_channel,0


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB


In [ ]:
df.describe()

,customer_id,zip
count,121930.000000,121930.000000
mean,78736.898663,50990.165595
std,45492.202886,26871.914605
min,1.000000,1001.000000
25%,39343.500000,28689.250000
50%,78784.500000,49835.000000
75%,118156.750000,73488.000000
max,157563.000000,99950.000000


# Định nghĩa các hàm dùng chung

In [ ]:
#thống kê và tìm kiếm các giá trị bất thường (outlier) bằng IQR
def iqr_outlier_stats(s):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty:
      return 0, 0.0, None, None
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = s[(s < lo) | (s > hi)]
    # Trả về 4 thông số: Số lượng outlier, tỷ lệ %, ngưỡng dưới và ngưỡng trên.
    return len(outliers), round(len(outliers) / len(s) * 100, 3), lo, hi

In [ ]:
#phát hiện các giá trị bất thường (outlier) bằng Z-score
from sklearn.preprocessing import StandardScaler
def get_zscore_outliers(s, thresh=3.0):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty or s.std(ddof=0) == 0: return 0, 0.0
    scaler = StandardScaler()
    z_scores = scaler.fit_transform(s.values.reshape(-1, 1))
    return pd.Series(np.abs(z_scores.flatten()) > thresh, index=s.index)
#index=s.index: gắn lại đúng tên dòng (nhãn chỉ mục) của bảng dữ liệu gốc cho cột kết quả
# flatten(): chuyển mảng 2 chiều thành mảng 1 chiều
# reshape(-1, 1): -1:chọn tất cả giá trị và 1:chuyển về mảng 1 chiều

In [ ]:
def normalize_person_name(x):
    """Trim khoảng trắng thừa (kể cả khoảng trắng ở giữa), viết hoa chữ cái đầu mỗi từ."""
    if pd.isna(x):
        return x
    x = " ".join(str(x).strip().split())
    return x.title() if x != "" else np.nan

In [ ]:
def normalize_zip_code(x):
    """Chỉ giữ lại các ký tự số và trả về dưới dạng chuỗi (text), không ép độ dài."""
    if pd.isna(x):
        return x
    # Ép kiểu sang str và lọc lấy các ký tự là số
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    # Trả về chuỗi kết quả nguyên bản
    return digits
def normalize_phone_vn(x, expected_len=9):
    """Số điện thoại VN chuẩn 10 số, bắt đầu bằng 0. Nếu nguồn chỉ có 9 số (mất số 0 đầu) thì bù lại."""
    if pd.isna(x):
        return x
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    if len(digits) == expected_len and not digits.startswith("0"):
        digits = "0" + digits
    return digits

In [ ]:
# 1. Rút gọn hàm làm sạch tiền tệ
def clean_currency_string(x):
    if pd.isna(x): return np.nan
    s = str(x)
    for tok in ['₫', 'VND', 'vnd', '$', 'USD', 'usd', ',', '%']:
        s = s.replace(tok, '')
    try:
        return float(s.strip())
    except ValueError:
        return np.nan
# Từ điển chữ số (có thể bổ sung thêm nếu cần)
WORD_NUMBER_MAP = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50
}
#Rút gọn hàm chuyển chữ/số thành float (dùng .get() thông minh hơn)
def words_or_number_to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    for tok in ["percent", "%%", "%"]:
        s = s.replace(tok, "")
    s = s.strip()
    # Tra cứu từ điển trước, nếu không có thì ép float, lỗi thì trả về NaN
    val = WORD_NUMBER_MAP.get(s)
    if val is None:
        try:
            val = float(s)
        except ValueError:
            return np.nan
    return val if val >= 0 else np.nan

# Kiểm tra trùng lặp và dữ liệu thiếu

In [ ]:
#Kiểm tra tỷ lệ thiếu ở từng cột
# Viết một hàm nhỏ để lấy giá trị đầu tiên không bị trống
def lay_gia_tri_dau(cot):
    hop_le = cot.dropna()
    if len(hop_le) > 0:
        return hop_le.iloc[0]  # Lấy giá trị đầu tiên
    return None                # Nếu cột toàn rỗng thì trả về None

# Sau đó dùng hàm này cho bảng
profile_df = pd.DataFrame({
    'Cột': df.columns,
    'Kiểu dữ liệu': df.dtypes.values,
    'Số lượng thiếu': df.isna().sum().values,
    'Tỷ lệ thiếu (%)': (df.isna().mean() * 100).round(2).values,
    'Số giá trị unique': df.nunique().values,
    'Ví dụ mẫu': [lay_gia_tri_dau(df[c]) for c in df.columns]
})
profile_df

,Cột,Kiểu dữ liệu,Số lượng thiếu,Tỷ lệ thiếu (%),Số giá trị unique,Ví dụ mẫu
0,customer_id,int64,0,0.0,121930,1
1,zip,int64,0,0.0,31491,15201
2,city,object,0,0.0,42,Hai Phong
3,signup_date,object,0,0.0,3941,2021-12-30
4,gender,object,0,0.0,3,Female
5,age_group,object,0,0.0,5,35-44
6,acquisition_channel,object,0,0.0,6,social_media


In [ ]:
cols = ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
for c in cols:
    if c in df.columns:
        so_luong = (df[c].value_counts().sort_index(ascending=True) > 1).sum()
        print(f"Số lượng {c} xuất hiện nhiều hơn 1 lần là: {so_luong}")


Số lượng customer_id xuất hiện nhiều hơn 1 lần là: 0
Số lượng zip xuất hiện nhiều hơn 1 lần là: 31169
Số lượng city xuất hiện nhiều hơn 1 lần là: 42
Số lượng signup_date xuất hiện nhiều hơn 1 lần là: 3888
Số lượng gender xuất hiện nhiều hơn 1 lần là: 3
Số lượng age_group xuất hiện nhiều hơn 1 lần là: 5
Số lượng acquisition_channel xuất hiện nhiều hơn 1 lần là: 6


In [ ]:
cols = ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
for c in cols:
    if c in df.columns:
        counts = df[c].value_counts()
        frequent_values = counts[counts > 1].index
        print(f"giá trị của cột {c} xuất hiện nhiều hơn 1 lần là:",list(frequent_values))

giá trị của cột customer_id xuất hiện nhiều hơn 1 lần là: []
giá trị của cột zip xuất hiện nhiều hơn 1 lần là: [92592, 93065, 90250, 68124, 68132, 68135, 68136, 68137, 68157, 68310, 68329, 68333, 68337, 44807, 44813, 44817, 44818, 68313, 44824, 68355, 68357, 68360, 68366, 68371, 68376, 68463, 68410, 68413, 68418, 68439, 44699, 44720, 68461, 44820, 68401, 68005, 44874, 44862, 44859, 44855, 72956, 72944, 72928, 72918, 72916, 72901, 72857, 72855, 72824, 72811, 68016, 68628, 68019, 68033, 68041, 68064, 68070, 68104, 72801, 68117, 44839, 44842, 44844, 44846, 44848, 68122, 68123, 68112, 44618, 44621, 68958, 68822, 68825, 68826, 68828, 68626, 68834, 68836, 68843, 68845, 68846, 68860, 68866, 68875, 68833, 44502, 44505, 69037, 69042, 44637, 68926, 68927, 44613, 68939, 68941, 68943, 68944, 44512, 44601, 44608, 44610, 68928, 44671, 68506, 68510, 68516, 68523, 68524, 68528, 68925, 44680, 44681, 44685, 44687, 44689, 44695, 44697, 49425, 44676, 44625, 44626, 68637, 68641, 68648, 68652, 68654, 44670,

In [ ]:
#Kiểm tra trùng lặp
n_dup_full = df.duplicated().sum()
n_dup_key = df.duplicated(subset=['customer_id']).sum()
print(f"Số dòng trùng lặp hoàn toàn: {n_dup_full}")
print(f"Số dòng trùng theo khoá chính customer_id: {n_dup_key}")
if n_dup_key > 0:
    df = df.drop_duplicates(subset=['customer_id'], keep='first')
print("Kích thước sau khi loại trùng:", df.shape)

Số dòng trùng lặp hoàn toàn: 0
Số dòng trùng theo khoá chính customer_id: 0
Kích thước sau khi loại trùng: (121930, 7)


# Chuẩn hoá định dạng & ép kiểu dữ liệu

In [ ]:
# Chuẩn hoá format
df['zip'] = df['zip'].apply(normalize_zip_code)
df['city'] = df['city'].apply(normalize_person_name)
for c in ['gender', 'acquisition_channel']:
    df[c] = df[c].str.strip().str.lower()
df['age_group'] = df['age_group'].str.strip()
print("Giá trị category sau chuẩn hoá:")
print('gender:', sorted(df['gender'].dropna().unique()))
print('acquisition_channel:', sorted(df['acquisition_channel'].dropna().unique()))
df[['zip', 'city']].head()

Giá trị category sau chuẩn hoá:
gender: ['female', 'male', 'non-binary']
acquisition_channel: ['direct', 'email_campaign', 'organic_search', 'paid_search', 'referral', 'social_media']


,zip,city
0,15201,Hai Phong
1,15201,Hai Phong
2,15201,Hai Phong
3,15201,Hai Phong
4,15201,Hai Phong


In [ ]:
#Chuyển kiểu
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce').astype('Int64')
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
for c in ['city', 'gender', 'age_group', 'acquisition_channel']:
    if c in df.columns: # Check if column exists before attempting to cast
        df[c] = df[c].astype('category')
print(df.dtypes)

customer_id                     Int64
zip                            object
city                         category
signup_date            datetime64[ns]
gender                       category
age_group                    category
acquisition_channel          category
dtype: object


# Kiểm tra logic nghiệp vụ cho từng cột

In [ ]:
# logic nghiệp vụ
max_date_observed = df['signup_date'].max()
n_future = (df['signup_date'] > pd.Timestamp('2022-12-31')).sum()
print(f"Số signup_date sau 2022-12-31 (ngoài phạm vi dữ liệu đơn hàng quan sát được): {n_future}")

Số signup_date sau 2022-12-31 (ngoài phạm vi dữ liệu đơn hàng quan sát được): 0


In [ ]:
# Kiểm tra logic nghiệp vụ cho từng cột
print("\n--- Kiểm tra cột 'customer_id' ---")
# customer_id: Đảm bảo là duy nhất và dương
if df['customer_id'].is_unique:
    print("customer_id: Tất cả các giá trị là duy nhất.")
else:
    print("customer_id: Có giá trị trùng lặp.")
if (df['customer_id'] > 0).all():
    print("customer_id: Tất cả các giá trị đều dương.")
else:
    print("customer_id: Có giá trị không dương.")
print("\n--- Kiểm tra cột 'gender' ---")
# gender: Đảm bảo nằm trong các danh mục hợp lệ
expected_genders = ['female', 'male', 'non-binary']
current_genders = df['gender'].dropna().unique().tolist()
if set(current_genders).issubset(set(expected_genders)):
    print(f"gender: Tất cả các giá trị đều nằm trong danh mục hợp lệ: {expected_genders}")
else:
    invalid_genders = set(current_genders) - set(expected_genders)
    print(f"gender: Có các giá trị không hợp lệ: {list(invalid_genders)}")
    print(f"Các giá trị hiện tại: {current_genders}")

print("\n--- Kiểm tra cột 'age_group' ---")
# age_group: Đảm bảo nằm trong các danh mục hợp lệ
expected_age_groups = ['18-24', '25-34', '35-44', '45-54', '55+']
current_age_groups = df['age_group'].dropna().unique().tolist()
if set(current_age_groups).issubset(set(expected_age_groups)):
    print(f"age_group: Tất cả các giá trị đều nằm trong danh mục hợp lệ: {expected_age_groups}")
else:
    invalid_age_groups = set(current_age_groups) - set(expected_age_groups)
    print(f"age_group: Có các giá trị không hợp lệ: {list(invalid_age_groups)}")
    print(f"Các giá trị hiện tại: {current_age_groups}")

print("\n--- Kiểm tra cột 'acquisition_channel' ---")
# acquisition_channel: Đảm bảo nằm trong các danh mục hợp lệ
expected_channels = ['social_media', 'email_campaign', 'organic_search', 'referral', 'paid_search', 'direct']
current_channels = df['acquisition_channel'].dropna().unique().tolist()
if set(current_channels).issubset(set(expected_channels)):
    print(f"acquisition_channel: Tất cả các giá trị đều nằm trong danh mục hợp lệ: {expected_channels}")
else:
    invalid_channels = set(current_channels) - set(expected_channels)
    print(f"acquisition_channel: Có các giá trị không hợp lệ: {list(invalid_channels)}")
    print(f"Các giá trị hiện tại: {current_channels}")



--- Kiểm tra cột 'customer_id' ---
customer_id: Tất cả các giá trị là duy nhất.
customer_id: Tất cả các giá trị đều dương.

--- Kiểm tra cột 'gender' ---
gender: Tất cả các giá trị đều nằm trong danh mục hợp lệ: ['female', 'male', 'non-binary']

--- Kiểm tra cột 'age_group' ---
age_group: Tất cả các giá trị đều nằm trong danh mục hợp lệ: ['18-24', '25-34', '35-44', '45-54', '55+']

--- Kiểm tra cột 'acquisition_channel' ---
acquisition_channel: Tất cả các giá trị đều nằm trong danh mục hợp lệ: ['social_media', 'email_campaign', 'organic_search', 'referral', 'paid_search', 'direct']


# Kiểm tra khóa ngoại

In [ ]:
geography_path = '/content/geography.csv'
geography_df = pd.read_csv(geography_path)
print(f"Kích thước bảng geography_df: {geography_df.shape}")
print("5 dòng đầu tiên của bảng geography_df:")
print(geography_df.head())
print("\nKiểu dữ liệu của các cột trong geography_df:")
print(geography_df.dtypes)

Kích thước bảng geography_df: (39948, 4)
5 dòng đầu tiên của bảng geography_df:
     zip       city region      district
0  15201  Hai Phong   East  District #13
1  15202     Phu Ly   East  District #13
2  15203   Viet Tri   East  District #13
3  15204  Bac Giang   East  District #13
4  15205  Bac Giang   East  District #13

Kiểu dữ liệu của các cột trong geography_df:
zip          int64
city        object
region      object
district    object
dtype: object


In [ ]:
#Kiểm tra khóa ngoại cho cột 'zip'
# Chuyển đổi chuỗi để đảm bảo so sánh đồng nhất kiểu dữ liệu
df['zip_str'] = df['zip'].astype(str)
geography_df['zip_str'] = geography_df['zip'].astype(str)
zips_not_in_master = df[~df['zip_str'].isin(geography_df['zip_str'])]
if not zips_not_in_master.empty:
    print(f"\nCó {len(zips_not_in_master)} dòng có `zip` trong `df` không tồn tại trong `geography_df`.")
    print("Các mã `zip` không khớp (5 ví dụ đầu):", zips_not_in_master['zip'].dropna().unique()[:5].tolist())
else:
    print("\nTất cả `zip` trong `df` đều khớp với `geography_df`.")
df = df.drop(columns=['zip_str'], errors='ignore')


Tất cả `zip` trong `df` đều khớp với `geography_df`.


# Xuất file kết quả

In [ ]:
df = df.drop(columns=['city'], errors='ignore')
# errors='raise': Nếu bảng df không có cột city, code sẽ lập tức văng lỗi (KeyError) và dừng chương trình.
df.to_csv('customers_silver.csv', index=False, encoding='utf-8-sig')
print("Đã lưu file đã làm sạch tại: customers_silver.csv",)
print(f"Kích thước cuối cùng: {df.shape}")

Đã lưu file đã làm sạch tại: customers_silver.csv
Kích thước cuối cùng: (121930, 6)
